# Task 03 – Graph Data Preparation & Preprocessing Pipeline

**Course Module:** CCS4354 – Tensors and Graphs  
**Dataset:** `ogbn-arxiv` Citation Benchmark  

---
### Required Deliverables & Workflow:
1. **Load Node Features:** Ingest 128-dimensional continuous text representations ($169,343 \times 128$).
2. **Load Labels:** Ingest 40-class ground truth subject categories ($169,343 \times 1$).
3. **Temporal Partitioning:** Construct realistic, out-of-distribution chronological splits:
   - **Training Set:** Papers published $\le 2017$
   - **Validation Set:** Papers published in $2018$
   - **Test Set:** Papers published in $2019\text{--}2020$
4. **Preprocessing Decisions & Normalization:** Provide theoretical justifications for graph symmetrization and feature normalization decisions.

In [1]:
import sys
from pathlib import Path

# Make the project root importable when running locally
PROJECT_ROOT = Path().resolve().parent if Path().resolve().name == 'notebooks' else Path().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import pandas as pd
import numpy as np

from src.config import RAW_DATA_DIR, PROCESSED_DATA_DIR
from src.data import load_ogbn_arxiv, prepare_and_save_data
from src.data.splits import split_summary

print("📥 Ingesting raw OGBN-Arxiv data...")
dataset, data, split_idx = load_ogbn_arxiv(RAW_DATA_DIR)
print(f"✅ Dataset loaded: {data.num_nodes:,} nodes, {data.num_edges:,} edges.")

📥 Ingesting raw OGBN-Arxiv data...


✅ Dataset loaded: 169,343 nodes, 2,315,598 edges.


C:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\ogb\nodeproppred\dataset_pyg.py:91: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  train_idx = torch.from_numpy(pd.read_csv(osp.join(path, 'train.csv.gz'), compression='gzip', header = None).values.T[0]).to(torch.long)


---
## 1. Node Features & Category Labels Inspection

- `data.x`: Feature tensor of shape `[169343, 128]` (float32 continuous values).
- `data.y`: Category labels of shape `[169343, 1]` spanning integers `0` to `39` corresponding to arXiv CS subject areas (e.g., `cs.CV`, `cs.LG`, `cs.AI`, `cs.CR`, `cs.NE`).

In [2]:
labels = data.y.squeeze().cpu().numpy()
unique_classes, class_counts = np.unique(labels, return_counts=True)

print(f"Feature Tensor Shape : {data.x.shape} (dtype: {data.x.dtype})")
print(f"Label Tensor Shape   : {data.y.shape} (dtype: {data.y.dtype})")
print(f"Total Target Classes : {len(unique_classes)} distinct categories")
print(f"Most Frequent Class  : Class {unique_classes[np.argmax(class_counts)]} ({class_counts.max():,} papers)")
print(f"Least Frequent Class : Class {unique_classes[np.argmin(class_counts)]} ({class_counts.min():,} papers)")

Feature Tensor Shape : torch.Size([169343, 128]) (dtype: torch.float32)
Label Tensor Shape   : torch.Size([169343, 1]) (dtype: torch.int64)
Total Target Classes : 40 distinct categories
Most Frequent Class  : Class 16 (27,321 papers)
Least Frequent Class : Class 12 (29 papers)


---
## 2. Temporal Split Partitioning

Standard random uniform splits create unrealistic data leakage in citation networks because future papers cite past papers. Therefore, we use **OGB's official temporal partitioning** based on publication year:

In [3]:
splits = split_summary(split_idx)
total_nodes = data.num_nodes

split_df = pd.DataFrame([
    {'Split Partition': 'Training Set', 'Publication Window': '≤ 2017', 'Paper Count': f"{splits['train']:,}", 'Percentage': f"{splits['train']/total_nodes*100:.2f}%", 'Purpose': 'Supervised parameter optimization'},
    {'Split Partition': 'Validation Set', 'Publication Window': '2018', 'Paper Count': f"{splits['valid']:,}", 'Percentage': f"{splits['valid']/total_nodes*100:.2f}%", 'Purpose': 'Hyperparameter tuning & early stopping'},
    {'Split Partition': 'Test Set', 'Publication Window': '2019 – 2020', 'Paper Count': f"{splits['test']:,}", 'Percentage': f"{splits['test']/total_nodes*100:.2f}%", 'Purpose': 'Unbiased inductive benchmark evaluation'}
])

print("📊 Official Chronological Partitioning Scorecard:")
display(split_df)

📊 Official Chronological Partitioning Scorecard:


,Split Partition,Publication Window,Paper Count,Percentage,Purpose
0,Training Set,≤ 2017,"90,941",53.70%,Supervised parameter optimization
1,Validation Set,2018,"29,799",17.60%,Hyperparameter tuning & early stopping
2,Test Set,2019 – 2020,"48,603",28.70%,Unbiased inductive benchmark evaluation


---
## 3. Preprocessing Decisions & Normalization Rationale

### Decision 1: Graph Symmetrization ($\mathbf{\tilde{A}} = \mathbf{A} + \mathbf{A}^\top + \mathbf{I}_N$)
- **Rationale:** Real citation networks are strictly directed (new papers only cite existing papers). In directed form, early papers have zero in-degree messages, leading to severe information isolation in GNN layers. Converting directed citations into undirected edges allows bidirectional semantic propagation.
- **Self-Loops ($\mathbf{I}_N$):** Ensures that every node retains its own feature representation during neighborhood message aggregation.

### Decision 2: Feature Normalization Rationale
- **Rationale:** The 128-dimensional Word2Vec skip-gram embeddings provided by OGB are already continuous, zero-centered, and unit-variance normalized across the semantic vocabulary. Hence, additional min-max or z-score feature scaling is unnecessary and could distort word vector angular distances.

In [4]:
print("⚙️ Executing Data Preprocessing and Serialization Pipeline...")
prepared_data = prepare_and_save_data(data, split_idx, PROCESSED_DATA_DIR, normalize=False)

print(f"\n✅ Preprocessed Data Saved to: {PROCESSED_DATA_DIR}")
print(f"   - Processed Features Shape : {prepared_data.x.shape}")
print(f"   - Processed Labels Shape   : {prepared_data.y.shape}")
print(f"   - Processed Edges Count    : {prepared_data.edge_index.shape[1]:,}")
print("🎉 Task 03 Data Preparation is 100% complete and ready for model training!")

⚙️ Executing Data Preprocessing and Serialization Pipeline...



✅ Preprocessed Data Saved to: D:\Projects\Done\Tenso Project\data\processed
   - Processed Features Shape : torch.Size([169343, 128])
   - Processed Labels Shape   : torch.Size([169343, 1])
   - Processed Edges Count    : 2,315,598
🎉 Task 03 Data Preparation is 100% complete and ready for model training!
